In [1]:
!unzip -q new_labeled_data.zip -d data
!find data -type f

data/cairo.txt
data/aruba.txt
data/tulum1.txt
data/milan.txt
data/tulum2.txt


In [2]:
!head -n 15 "data/aruba.txt"
!wc -l "data/aruba.txt"

2010-11-04 00:03:50.209589 M003 ON Sleeping begin
2010-11-04 00:03:57.399391 M003 OFF
2010-11-04 00:15:08.984841 T002 21.5
2010-11-04 00:30:19.185547 T003 21
2010-11-04 00:30:19.385336 T004 21
2010-11-04 00:35:22.24587 T005 20.5
2010-11-04 00:40:25.428962 T005 21
2010-11-04 00:45:28.658171 T005 20.5
2010-11-04 01:05:42.269469 T001 20
2010-11-04 01:15:48.936777 T002 21
2010-11-04 01:30:59.100184 T003 20.5
2010-11-04 01:46:09.301846 T005 20
2010-11-04 01:56:16.059687 T004 20.5
2010-11-04 02:21:32.138203 T001 19.5
2010-11-04 02:21:32.228625 T002 20.5
1719558 data/aruba.txt


In [3]:
import pandas as pd

rows = []
with open("data/aruba.txt") as f:
    for line in f:
        parts = line.split()
        if len(parts) < 3:          # skip broken or empty lines
            continue
        rows.append({
            "time":   parts[0] + " " + parts[1],
            "sensor": parts[2],
            "value":  parts[3] if len(parts) > 3 else "",
            "label":  " ".join(parts[4:]),   # e.g. "Sleeping begin" or ""
        })

df = pd.DataFrame(rows)
df["time"] = pd.to_datetime(df["time"], format="mixed", errors="coerce")

print(df.shape)
print("Bad timestamps:", df["time"].isna().sum())
print(df["time"].min(), "->", df["time"].max())
print(df["sensor"].str[0].value_counts())      # M / D / T counts
print(df[df["label"] != ""]["label"].value_counts())
df.head()

(1719558, 4)
Bad timestamps: 0
2010-11-04 00:03:50.209589 -> 2011-06-11 23:58:10.004834
sensor
M    1596145
T     116572
D       6836
L          4
E          1
Name: count, dtype: int64
label
Relax end                 2919
Relax begin               2919
Meal_Preparation begin    1606
Meal_Preparation end      1606
Leave_Home begin           431
Leave_Home end             431
Enter_Home end             431
Enter_Home begin           431
Sleeping begin             401
Sleeping end               401
Eating end                 257
Eating begin               257
Work begin                 171
Work end                   171
Bed_to_Toilet begin        157
Bed_to_Toilet end          157
Wash_Dishes end             65
Wash_Dishes begin           65
Housekeeping begin          33
Housekeeping end            33
Respirate begin              6
Respirate end                6
Name: count, dtype: int64


,time,sensor,value,label
0,2010-11-04 00:03:50.209589,M003,ON,Sleeping begin
1,2010-11-04 00:03:57.399391,M003,OFF,
2,2010-11-04 00:15:08.984841,T002,21.5,
3,2010-11-04 00:30:19.185547,T003,21,
4,2010-11-04 00:30:19.385336,T004,21,


In [4]:
# activities can overlap, so I keep a list of the ones still open
stack = []
acts = []
overlaps = 0

for lab in df["label"]:
    if lab.endswith(" begin"):
        if stack:
            overlaps += 1
        stack.append(lab[:-6])
        acts.append(stack[-1])
    elif lab.endswith(" end"):
        name = lab[:-4]
        acts.append(name)
        if name in stack:
            stack.remove(name)
    else:
        acts.append(stack[-1] if stack else None)

df["activity"] = pd.Series(acts).fillna("Other")

# keep motion (M) and door (D) sensors only, after labelling,
# because a begin/end label can be on any sensor line
df = df[df["sensor"].str[0].isin(["M", "D"])].reset_index(drop=True)

print("Overlaps:", overlaps)
print("Still open at the end:", stack)
print(df["activity"].value_counts())

Overlaps: 58
Still open at the end: []
activity
Other               871302
Relax               349241
Meal_Preparation    289772
Sleeping             32728
Eating               16746
Work                 16321
Housekeeping         10579
Wash_Dishes          10462
Enter_Home            2026
Leave_Home            1932
Bed_to_Toilet         1330
Respirate              542
Name: count, dtype: int64


In [5]:
# 1) Merge the tiny class into "Other"
df["activity"] = df["activity"].replace("Respirate", "Other")

# 2) Cut time into 5-minute windows
df["window"] = df["time"].dt.floor("5min")

# 3) Features: how many times each sensor fired in each window
X = pd.crosstab(df["window"], df["sensor"])

# 4) Label: the most common activity among the events in that window
y = df.groupby("window")["activity"].agg(lambda s: s.value_counts().idxmax())

print("Windows:", X.shape[0], " Sensors:", X.shape[1])
print(X.iloc[:3, :8])
print(y.value_counts())
print("Majority baseline:", round(y.value_counts(normalize=True).max(), 3))

Windows: 32593  Sensors: 34
sensor               D001  D002  D004  M001  M002  M003  M004  M005
window                                                             
2010-11-04 00:00:00     0     0     0     0     0     2     0     0
2010-11-04 02:30:00     0     0     0     0     0     2     0     0
2010-11-04 03:40:00     0     0     0     0     0     2     0     0
activity
Relax               13770
Other               11855
Sleeping             2924
Meal_Preparation     2883
Work                  504
Eating                380
Housekeeping          128
Wash_Dishes           114
Enter_Home             16
Leave_Home             14
Bed_to_Toilet           5
Name: count, dtype: int64
Majority baseline: 0.422


In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 1) Merge the tiniest classes into "Other"
y2 = y.replace(["Bed_to_Toilet", "Enter_Home", "Leave_Home"], "Other")

# 2) Split by time: first 80% train, last 20% test
cut = X.index.min() + (X.index.max() - X.index.min()) * 0.8
is_train = X.index < cut
X_train, X_test = X[is_train], X[~is_train]
y_train, y_test = y2[is_train], y2[~is_train]
print("Train windows:", len(X_train), " Test windows:", len(X_test))
print(y_test.value_counts())

# 3) Scale the features (fit on train only)
scaler = StandardScaler().fit(np.log1p(X_train))
Xtr = scaler.transform(np.log1p(X_train))
Xte = scaler.transform(np.log1p(X_test))

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"{name:24s} accuracy={acc:.3f}  macro-F1={f1:.3f}")

# 4) Baseline A: always the most common label
majority = y_train.value_counts().idxmax()
report("Majority", y_test, [majority] * len(y_test))

# 5) Baseline B: hour of day only
best_by_hour = y_train.groupby(X_train.index.hour).agg(lambda s: s.value_counts().idxmax())
pred_hour = pd.Series(X_test.index.hour, index=X_test.index).map(best_by_hour).fillna(majority)
report("Hour of day only", y_test, pred_hour)

# 6) Logistic regression
lr = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
report("Logistic regression", y_test, lr.predict(Xte))

# 7) kNN
knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y_train)
report("kNN (k=15)", y_test, knn.predict(Xte))

Train windows: 26425  Test windows: 6168
activity
Relax               2465
Other               2393
Meal_Preparation     624
Sleeping             529
Work                  91
Eating                59
Wash_Dishes            7
Name: count, dtype: int64
Majority                 accuracy=0.400  macro-F1=0.082
Hour of day only         accuracy=0.496  macro-F1=0.223
Logistic regression      accuracy=0.871  macro-F1=0.679
kNN (k=15)               accuracy=0.875  macro-F1=0.675


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

pred = lr.predict(Xte)
labels = sorted(y2.unique())

print("Housekeeping in train:", (y_train == "Housekeeping").sum(),
      " in test:", (y_test == "Housekeeping").sum())
print()
print(classification_report(y_test, pred, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=labels),
                   index=labels, columns=labels))

Housekeeping in train: 128  in test: 0

                  precision    recall  f1-score   support

          Eating       0.46      0.56      0.50        59
Meal_Preparation       0.85      0.75      0.79       624
           Other       0.81      0.88      0.84      2393
           Relax       0.94      0.89      0.92      2465
        Sleeping       0.93      0.95      0.94       529
     Wash_Dishes       0.00      0.00      0.00         7
            Work       0.88      0.66      0.75        91

        accuracy                           0.87      6168
       macro avg       0.70      0.67      0.68      6168
    weighted avg       0.87      0.87      0.87      6168

                  Eating  Housekeeping  Meal_Preparation  Other  Relax  \
Eating                33             0                 3     22      1   
Housekeeping           0             0                 0      0      0   
Meal_Preparation       1             0               466    153      2   
Other                 3

In [8]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)

# 1) Merge the two rare classes into Other
y3 = y2.replace(["Housekeeping", "Wash_Dishes"], "Other")
y_train, y_test = y3[is_train], y3[~is_train]
classes = sorted(y3.unique())
to_idx = {c: i for i, c in enumerate(classes)}
ytr = torch.tensor(y_train.map(to_idx).values)
print(classes)

# 2) Re-run the baselines with the new labels
majority = y_train.value_counts().idxmax()
report("Majority", y_test, [majority] * len(y_test))
lr = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
report("Logistic regression", y_test, lr.predict(Xte))

# 3) The network, built in two parts
n_in, n_cls = Xtr.shape[1], len(classes)
home_part   = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU())
server_part = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, n_cls))
model = nn.Sequential(home_part, server_part)

# 4) Train for a fixed 30 epochs
Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)
loader = DataLoader(TensorDataset(Xtr_t, ytr), batch_size=256, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 31):
    model.train()
    total = 0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  train loss {total/len(Xtr_t):.4f}")

# 5) Look at the test set once
model.eval()
with torch.no_grad():
    pred_idx = model(Xte_t).argmax(dim=1).numpy()
pred_nn = [classes[i] for i in pred_idx]
report("Neural net (centralized)", y_test, pred_nn)

['Eating', 'Meal_Preparation', 'Other', 'Relax', 'Sleeping', 'Work']
Majority                 accuracy=0.400  macro-F1=0.095
Logistic regression      accuracy=0.871  macro-F1=0.790
epoch  5  train loss 0.2941
epoch 10  train loss 0.2709
epoch 15  train loss 0.2591
epoch 20  train loss 0.2526
epoch 25  train loss 0.2466
epoch 30  train loss 0.2427
Neural net (centralized) accuracy=0.884  macro-F1=0.810


In [9]:
import os

def make_parts(seed):
    torch.manual_seed(seed)
    home = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU())
    server = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, n_cls))
    return home, server

def make_loader(seed):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(Xtr_t, ytr), batch_size=256, shuffle=True, generator=g)

def evaluate(home, server):
    home.eval(); server.eval()
    with torch.no_grad():
        idx = server(home(Xte_t)).argmax(dim=1).numpy()
    return [classes[i] for i in idx]

def train_central(seed, epochs=30):
    home, server = make_parts(seed)
    loader = make_loader(seed)
    opt = torch.optim.Adam(list(home.parameters()) + list(server.parameters()), lr=1e-3)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(server(home(xb)), yb).backward()
            opt.step()
    return home, server

def train_split(seed, epochs=30):
    home, server = make_parts(seed)
    loader = make_loader(seed)
    opt_home = torch.optim.Adam(home.parameters(), lr=1e-3)
    opt_server = torch.optim.Adam(server.parameters(), lr=1e-3)
    sent = 0                                   # bytes that travel
    for _ in range(epochs):
        for xb, yb in loader:
            # HOME: forward through its own part only
            opt_home.zero_grad()
            act = home(xb)
            # message 1: home -> server (activations and labels)
            act_server = act.detach().requires_grad_()
            sent += act.numel() * 4 + yb.numel() * 8
            # SERVER: finish forward, loss, backward through its part
            opt_server.zero_grad()
            loss = loss_fn(server(act_server), yb)
            loss.backward()
            opt_server.step()
            # message 2: server -> home (gradient for the activations)
            grad = act_server.grad
            sent += grad.numel() * 4
            # HOME: continue backward with the received gradient
            act.backward(grad)
            opt_home.step()
    return home, server, sent

rows = []
for seed in [0, 1, 2]:
    h, s = train_central(seed)
    p = evaluate(h, s)
    rows.append(["centralized", seed, accuracy_score(y_test, p),
                 f1_score(y_test, p, average="macro", zero_division=0), 0.0])
    h, s, sent = train_split(seed)
    p = evaluate(h, s)
    rows.append(["split", seed, accuracy_score(y_test, p),
                 f1_score(y_test, p, average="macro", zero_division=0), sent / 1e6])

res = pd.DataFrame(rows, columns=["method", "seed", "accuracy", "macro_F1", "MB_sent_in_training"])
print(res.round(3))
print()
print(res.groupby("method")[["accuracy", "macro_F1"]].agg(["mean", "std"]).round(3))

upload_mb = (Xtr_t.numel() * 4 + ytr.numel() * 8) / 1e6
print(f"\nCentralized: one-time upload of the training table = {upload_mb:.1f} MB")
print(f"Raw event file (all 7 months, for scale)            = {os.path.getsize('data/aruba.txt')/1e6:.1f} MB")

        method  seed  accuracy  macro_F1  MB_sent_in_training
0  centralized     0     0.885     0.810                 0.00
1        split     0     0.885     0.810               412.23
2  centralized     1     0.884     0.815                 0.00
3        split     1     0.884     0.815               412.23
4  centralized     2     0.880     0.794                 0.00
5        split     2     0.880     0.794               412.23

            accuracy        macro_F1       
                mean    std     mean    std
method                                     
centralized    0.883  0.002    0.806  0.011
split          0.883  0.002    0.806  0.011

Centralized: one-time upload of the training table = 3.8 MB
Raw event file (all 7 months, for scale)            = 61.1 MB


In [10]:
best_by_hour = y_train.groupby(X_train.index.hour).agg(lambda s: s.value_counts().idxmax())
pred_hour = pd.Series(X_test.index.hour, index=X_test.index).map(best_by_hour).fillna(majority)
report("Hour of day only", y_test, pred_hour)

knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y_train)
report("kNN (k=15)", y_test, knn.predict(Xte))

Hour of day only         accuracy=0.496  macro-F1=0.261
kNN (k=15)               accuracy=0.874  macro-F1=0.786
